# TrOCR fine-tuning on RUKOPYS (Mavka team)

Runtime → Change runtime type → **T4 GPU** (or better).

**Where things live**
- Dataset (19.3 GB) → Colab's local disk `/content/rukopys`. It is **not** copied to Drive.
- Line crops → local disk, plus one archive copied to Drive so a new session can skip the download.
- Checkpoints and final models → Drive, so training survives disconnects.

**Order:** cell 1 → Runtime → Restart session → cells 2–6. After training, cell 7.
If Colab disconnects: the same order again. Cell 3 restores crops from Drive, and cell 6 resumes from the last checkpoint.

## 1. GPU, disk, packages

In [ ]:
!nvidia-smi --query-gpu=name,memory.total --format=csv
!df -h /content | tail -1
# Pinned to transformers 4.x because this notebook uses Seq2SeqTrainer and VisionEncoderDecoderModel.
!pip -q install "transformers>=4.45,<5" "accelerate>=0.34" evaluate jiwer huggingface_hub

name, memory.total [MiB]
Tesla T4, 15360 MiB
overlay         113G   83M  113G   1% /
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
gradio-client 2.7.2 requires huggingface-hub<3.0,>=1.16.0, but you have huggingface-hub 0.36.2 which is incompatible.
gradio 6.31.0 requires huggingface-hub<3.0,>=1.16.0, but you have huggingface-hub 0.36.2 which is incompatible.
diffusers 0.42.1 requires huggingface-hub<3.0,>=1.32.0, but you have huggingface-hub 0.36.2 which is incompatible.


## 2. Paths and settings

In [1]:
from google.colab import drive
drive.mount('/content/drive')

from pathlib import Path
DATA  = Path('/content/rukopys')                      # raw dataset, local disk only
CROPS = Path('/content/crops')                        # line crops, local disk
OUT   = Path('/content/drive/MyDrive/rukopys_trocr')  # checkpoints + models + crop archive
OUT.mkdir(parents=True, exist_ok=True)
CROPS_TAR = OUT / 'crops.tar'

BASE_MODEL  = 'Hukyl/trocr-large-uk-handwritten'     # zero-shot CER 0.158; '-real' variant scored the same
USE_SILVER  = False       # True only if your crops.tar includes silver (made without --no-silver)
SILVER_LIMIT = None       # e.g. 100_000 to shorten stage A on a free T4
VAL_FRAC, SEED = 0.2, 42
MAX_W = 1536              # crops wider than this are downscaled (TrOCR resizes to 384x384 anyway)

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


## 3. Restore crops from Drive (skip download if they exist)

In [2]:
import tarfile, os, shutil
if CROPS_TAR.exists() and CROPS_TAR.stat().st_size < 10_000_000:
    print('Removing an empty/broken crop archive from Drive'); CROPS_TAR.unlink()
gt = CROPS / 'gold_train.jsonl'
if CROPS_TAR.exists() and not (gt.exists() and gt.stat().st_size > 0):
    shutil.rmtree(CROPS, ignore_errors=True)
    print('Restoring crops from Drive...')
    with tarfile.open(CROPS_TAR) as t: t.extractall('/content')
HAVE_CROPS = (CROPS / 'gold_train.jsonl').exists() and (CROPS / 'gold_train.jsonl').stat().st_size > 0
print('crops ready:', HAVE_CROPS)

crops ready: True


## 4. Download the dataset (only if crops are not ready)
If the download asks for authorization, accept the dataset terms on its Hugging Face page,
then add an `HF_TOKEN` secret in Colab (key icon on the left) with notebook access enabled.

In [3]:
from huggingface_hub import snapshot_download
if not HAVE_CROPS:
    try:
        from google.colab import userdata
        os.environ.setdefault('HF_TOKEN', userdata.get('HF_TOKEN') or '')
    except Exception:
        pass
    patterns = ['train/*', 'test/*', '*.csv'] + (['silver/*'] if USE_SILVER else [])
    from huggingface_hub import HfApi
    from collections import Counter
    files = HfApi().list_repo_files('UkrainianCatholicUniversity/rukopys', repo_type='dataset')
    print('files in repo:', len(files))
    print(Counter('/'.join(f.split('/')[:2]) if f.count('/') > 1 else f for f in files).most_common(15))
    print('metadata files:', [f for f in files if f.endswith('.jsonl')])
    snapshot_download('UkrainianCatholicUniversity/rukopys', repo_type='dataset',
                      local_dir=str(DATA), allow_patterns=patterns, max_workers=8)
    found = sorted(str(p.relative_to(DATA)) for p in DATA.rglob('metadata.jsonl'))
    print('downloaded metadata:', found)
    assert (DATA / 'train' / 'metadata.jsonl').exists(), 'train/metadata.jsonl missing - send this output'
!du -sh /content/rukopys/* 2>/dev/null; df -h /content | tail -1

overlay         113G  4.4G  109G   4% /


## 5. Cut line crops + page-level train/val split
Keeps `handwritten`, `printed`, `annotation` with `language=uk` and `legibility=legible`.
`image`/`graph` (no text, not scored for CER) and `formula`/`table` (Qwen branch) are skipped.
Labels are normalized the same way the competition metric normalizes text.

In [4]:
import json, random, re, shutil
from PIL import Image, ImageOps
from tqdm.auto import tqdm

TEXT_TYPES = {'handwritten', 'printed', 'annotation'}
_STRIKE_FIX = re.compile(r'~~(.*?)~~\{(.*?)\}')
_STRIKE = re.compile(r'~~(.*?)~~')
_TABLE = str.maketrans({'—': '-', '–': '-', '«': '"', '»': '"', '“': '"', '”': '"', '„': '"',
                        '‘': "'", '’': "'", 'c': 'с', 'o': 'о', 'p': 'р', 'x': 'х'})
def normalize(t):
    t = _STRIKE_FIX.sub(r'\2', t or '')   # ~~old~~{new} -> new
    t = _STRIKE.sub(r'\1', t)             # ~~text~~ -> text
    return ' '.join(t.translate(_TABLE).split())

def keep(r):
    if r.get('type') not in TEXT_TYPES or r.get('language', 'uk') != 'uk': return False
    if r.get('legibility') == 'illegible': return False
    t = normalize(r.get('text'))
    if not t or t.startswith('[ERROR:'): return False
    x1, y1, x2, y2 = r['bbox']
    return (x2 - x1) >= 4 and (y2 - y1) >= 4

def load_page(path, W, H):
    im = Image.open(path)
    if W and H and im.size != (W, H):          # phone photos: bboxes may refer to EXIF-rotated image
        t = ImageOps.exif_transpose(im)
        if t.size == (W, H): im = t
    return im.convert('RGB')

def crop_split(split):
    meta = DATA / split / 'metadata.jsonl'
    if not meta.exists(): print('no', meta); return []
    out = CROPS / split; out.mkdir(parents=True, exist_ok=True)
    rows = []
    for line in tqdm(meta.read_text(encoding='utf-8').splitlines(), desc=split):
        page = json.loads(line)
        regs = [r for r in (page.get('regions') or []) if keep(r)]
        if not regs: continue
        fn = page.get('file_name') or page.get('image')
        try:
            im = load_page(DATA / split / fn, page.get('image_width'), page.get('image_height'))
        except Exception as e:
            print('skip', fn, e); continue
        stem = Path(fn).stem
        for i, r in enumerate(regs):
            x1, y1, x2, y2 = map(int, r['bbox'])
            c = im.crop((max(0, x1 - 4), max(0, y1 - 4), min(im.width, x2 + 4), min(im.height, y2 + 4)))
            if c.width > MAX_W:
                c = c.resize((MAX_W, max(1, round(c.height * MAX_W / c.width))))
            p = out / f'{stem}_{i}.jpg'
            c.save(p, quality=92)
            rows.append({'path': str(p.relative_to(CROPS)), 'text': normalize(r['text']),
                         'page': stem, 'type': r['type']})
    return rows

def dump(rows, name):
    with open(CROPS / name, 'w', encoding='utf-8') as f:
        for r in rows: f.write(json.dumps(r, ensure_ascii=False) + '\n')
    print(f'{name}: {len(rows)} crops')

if not HAVE_CROPS:
    CROPS.mkdir(exist_ok=True)
    gold = crop_split('train')
    assert gold, 'No gold crops: the dataset was not downloaded correctly (see cell 4 output).'
    pages = sorted({r['page'] for r in gold}); random.Random(SEED).shuffle(pages)
    val_pages = set(pages[:int(len(pages) * VAL_FRAC)])
    dump([r for r in gold if r['page'] not in val_pages], 'gold_train.jsonl')
    dump([r for r in gold if r['page'] in val_pages], 'gold_val.jsonl')
    dump(crop_split('test'), 'test.jsonl')   # final check only (empty if test has no labels)
    if USE_SILVER:
        dump(crop_split('silver'), 'silver.jsonl')
        shutil.rmtree(DATA / 'silver' / 'images', ignore_errors=True)   # free ~16 GB local disk
    with tarfile.open(CROPS_TAR, 'w') as t:   # backup to Drive (JPEGs: no point compressing)
        t.add(CROPS, arcname='crops')
    print('crop archive on Drive:', round(CROPS_TAR.stat().st_size / 1e9, 2), 'GB')
!df -h /content | tail -1

overlay         113G  4.4G  109G   4% /


## 6. Train (fine-tune Hukyl TrOCR-large on gold)
Zero-shot CER of this model on our validation lines was ~0.158. The cell first deletes the old
kazars24 run (`stageA`, unusable byte-level tokenizer) from Drive to free space.
Rerunning this cell resumes from the last checkpoint of `RUN_NAME`.

In [5]:
import os; os.environ['PYTORCH_CUDA_ALLOC_CONF'] = 'expandable_segments:True'
RUN_NAME = 'trocr_large_gold'
EPOCHS, LR = 4, 2e-5
BS, ACCUM = 4, 4            # effective batch 16; if CUDA OOM: BS, ACCUM = 2, 8
MAX_LEN = 192               # tokens (this tokenizer is ~1 token per character)
EVAL_SAVE_STEPS = 500
AUG_P = 0.35                # mild augmentation probability (as in the 3rd-place writeup)

import torch, evaluate, io
from PIL import ImageEnhance, ImageFilter
from transformers import (TrOCRProcessor, VisionEncoderDecoderModel, Seq2SeqTrainer,
                          Seq2SeqTrainingArguments, default_data_collator)
from transformers.trainer_utils import get_last_checkpoint
from transformers.models.vision_encoder_decoder.modeling_vision_encoder_decoder import shift_tokens_right

# Checkpoints go to Colab's local disk: files deleted on Drive go to the Drive trash and
# keep using your quota, and the Trainer deletes an old ~6.7 GB checkpoint at every save.
# Only the final model (~2.2 GB) is copied to Drive.
CKPT = Path('/content/runs') / RUN_NAME      # local checkpoints (lost if the runtime is deleted)
RUN = OUT / RUN_NAME                         # Drive: final model goes to RUN / 'final' (cell 7 reads it)

def read(name):
    p = CROPS / f'{name}.jsonl'
    return [json.loads(l) for l in p.read_text(encoding='utf-8').splitlines()] if p.exists() else []
train_rows = read('gold_train') + (read('silver') if USE_SILVER else [])
val_rows = read('gold_val')
val_small = random.Random(SEED).sample(val_rows, min(500, len(val_rows)))   # ~4-5 min per eval
print(f'train {len(train_rows)}  val {len(val_rows)}')

processor = TrOCRProcessor.from_pretrained(BASE_MODEL)
tok = processor.tokenizer
model = VisionEncoderDecoderModel.from_pretrained(BASE_MODEL)
print('"привіт" ->', tok.tokenize('привіт'))

# Keep the model's own special-token setup; fill in only what is missing.
if model.config.pad_token_id is None: model.config.pad_token_id = tok.pad_token_id
if model.config.decoder_start_token_id is None:
    model.config.decoder_start_token_id = model.generation_config.decoder_start_token_id
g = model.generation_config
g.max_length, g.num_beams, g.no_repeat_ngram_size, g.length_penalty, g.early_stopping = MAX_LEN, 1, 0, 1.0, False
print('decoder_start', model.config.decoder_start_token_id, '| bos', tok.bos_token_id,
      '| pad', model.config.pad_token_id, '| eos', tok.eos_token_id)

WITH_BOS = False   # decided by the pre-training check below

def make_labels(text):
    ids = tok(text, add_special_tokens=False).input_ids[:MAX_LEN - 2]
    ids = ([tok.bos_token_id] if WITH_BOS else []) + ids + [tok.eos_token_id]
    return ids + [-100] * (MAX_LEN - len(ids))

def augment(im, rng):
    if rng.random() > AUG_P: return im
    if rng.random() < 0.5:
        im = im.rotate(rng.uniform(-2, 2), expand=True, fillcolor=(255, 255, 255))
    if rng.random() < 0.5:
        im = ImageEnhance.Brightness(im).enhance(rng.uniform(0.8, 1.2))
        im = ImageEnhance.Contrast(im).enhance(rng.uniform(0.8, 1.2))
    if rng.random() < 0.3:
        im = im.filter(ImageFilter.GaussianBlur(rng.uniform(0.3, 1.0)))
    if rng.random() < 0.3:
        buf = io.BytesIO(); im.save(buf, 'JPEG', quality=rng.randint(40, 80)); im = Image.open(buf).convert('RGB')
    return im

class Crops(torch.utils.data.Dataset):
    def __init__(self, rows, train=False): self.rows, self.train = rows, train
    def __len__(self): return len(self.rows)
    def __getitem__(self, i):
        r = self.rows[i]
        im = Image.open(CROPS / r['path']).convert('RGB')
        if self.train: im = augment(im, random.Random())
        pix = processor(images=im, return_tensors='pt').pixel_values[0]
        return {'pixel_values': pix, 'labels': torch.tensor(make_labels(r['text']))}

# Plain next-token cross-entropy. The library's own loss for this model (ForCausalLMLoss)
# shifts the labels a second time, which is why the loss was ~15.
def ocr_loss(model, pixel_values, labels):
    dec_in = shift_tokens_right(labels, model.config.pad_token_id, model.config.decoder_start_token_id)
    logits = model(pixel_values=pixel_values, decoder_input_ids=dec_in).logits.float()
    return torch.nn.functional.cross_entropy(logits.reshape(-1, logits.size(-1)), labels.reshape(-1),
                                             ignore_index=-100)

class OCRTrainer(Seq2SeqTrainer):
    def __init__(self, *a, **k):
        super().__init__(*a, **k)
        self.model_accepts_loss_kwargs = False   # our loss is a per-batch mean -> Trainer divides by ACCUM
    def compute_loss(self, model, inputs, return_outputs=False, **kwargs):
        loss = ocr_loss(model, inputs['pixel_values'], inputs['labels'])
        return (loss, None) if return_outputs else loss

# ---- Pre-training check: the untrained model already reads Ukrainian (CER 0.156),
# so with the correct label format its loss must be low. Try both formats, keep the better one.
model.cuda().eval()
check_rows = val_small[:8]
pix = torch.stack([processor(images=Image.open(CROPS / r['path']).convert('RGB'),
                             return_tensors='pt').pixel_values[0] for r in check_rows]).cuda()
losses = {}
with torch.no_grad():
    for flag in ((False, True) if tok.bos_token_id is not None else (False,)):
        WITH_BOS = flag
        lab = torch.tensor([make_labels(r['text']) for r in check_rows]).cuda()
        losses[flag] = ocr_loss(model, pix, lab).item()
WITH_BOS = min(losses, key=losses.get)
print(f'pre-training loss check: without <s> = {losses[False]:.3f} | with <s> = {losses.get(True, float("nan")):.3f} '
      f'-> using WITH_BOS={WITH_BOS}')
del pix, lab; torch.cuda.empty_cache()
assert losses[WITH_BOS] < 3, 'Loss too high with both label formats. Do NOT train - send this output.'
model.train()

cer_metric = evaluate.load('cer')
def compute_metrics(pred):
    p, l = pred.predictions, pred.label_ids
    p[p == -100] = tok.pad_token_id; l[l == -100] = tok.pad_token_id
    preds = [normalize(s) for s in tok.batch_decode(p, skip_special_tokens=True)]
    refs = [normalize(s) for s in tok.batch_decode(l, skip_special_tokens=True)]
    return {'cer': cer_metric.compute(predictions=preds, references=refs)}

real_bf16 = torch.cuda.is_available() and torch.cuda.get_device_capability()[0] >= 8   # T4 = 7.5 -> fp16
args = Seq2SeqTrainingArguments(
    output_dir=str(CKPT), num_train_epochs=EPOCHS, learning_rate=LR,
    per_device_train_batch_size=BS, gradient_accumulation_steps=ACCUM, per_device_eval_batch_size=4,
    gradient_checkpointing=True,
    warmup_ratio=0.05, weight_decay=0.01, lr_scheduler_type='cosine', max_grad_norm=1.0,
    fp16=not real_bf16, bf16=real_bf16,
    eval_strategy='steps', eval_steps=EVAL_SAVE_STEPS,
    save_strategy='steps', save_steps=EVAL_SAVE_STEPS, save_total_limit=2,
    load_best_model_at_end=True, metric_for_best_model='cer', greater_is_better=False,
    predict_with_generate=True, generation_num_beams=1, generation_max_length=MAX_LEN,
    dataloader_num_workers=2, logging_steps=50, report_to='none',
)
trainer = OCRTrainer(model=model, args=args, train_dataset=Crops(train_rows, train=True),
                     eval_dataset=Crops(val_small), data_collator=default_data_collator,
                     compute_metrics=compute_metrics)
last = get_last_checkpoint(str(CKPT)) if CKPT.exists() else None
print('resuming from', last)
trainer.train(resume_from_checkpoint=last)

final = RUN / 'final'          # on Drive
trainer.save_model(str(final)); processor.save_pretrained(str(final))
print('saved best model to', final)

train 17779  val 4333


/usr/local/lib/python3.13/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(
Using a slow image processor as `use_fast` is unset and a slow processor was saved with this model. `use_fast=True` will be the default behavior in v4.52, even if the model was saved with a slow processor. This will result in minor differences in outputs. You'll still be able to use a slow processor with `use_fast=False`.


"привіт" -> ['п', 'р', 'и', 'в', 'і', 'т']
decoder_start 0 | bos 0 | pad 1 | eos 2
pre-training loss check: without <s> = 1.761 | with <s> = 0.353 -> using WITH_BOS=True
resuming from None


Step,Training Loss,Validation Loss,Cer
500,0.388200,10.169432,0.152689
1000,0.289700,10.494949,0.113738
1500,0.243200,11.523493,0.099512


`loss_type=None` was set in the config but it is unrecognized. Using the default loss: `ForCausalLMLoss`.


Step,Training Loss,Validation Loss,Cer
500,0.388200,10.169432,0.152689
1000,0.289700,10.494949,0.113738
1500,0.243200,11.523493,0.099512
2000,0.190000,11.759870,0.108657
2500,0.165500,12.480822,0.089351
3000,0.134600,12.716639,0.095854
3500,0.120400,13.651218,0.097074
4000,0.097400,13.750172,0.094296


There were missing keys in the checkpoint model loaded: ['decoder.output_projection.weight'].


saved best model to /content/drive/MyDrive/rukopys_trocr/trocr_large_gold/final


## 7. Full validation CER (greedy) + examples

In [6]:
del trainer; import gc; gc.collect(); torch.cuda.empty_cache()
model = VisionEncoderDecoderModel.from_pretrained(str(RUN / 'final')).cuda().eval().half()

@torch.no_grad()
def recognize(images, bs=32):
    out = []
    for i in range(0, len(images), bs):
        pix = processor(images=images[i:i + bs], return_tensors='pt').pixel_values.cuda().half()
        ids = model.generate(pix, max_length=MAX_LEN, num_beams=1, no_repeat_ngram_size=0)
        out += [normalize(s) for s in tok.batch_decode(ids, skip_special_tokens=True)]
    return out

def evaluate_rows(rows, name):
    if not rows: return
    preds = recognize([Image.open(CROPS / r['path']).convert('RGB') for r in rows])
    refs = [r['text'] for r in rows]
    print(f'{name}: CER = {cer_metric.compute(predictions=preds, references=refs):.4f}  ({len(rows)} crops)')
    for t in sorted({r['type'] for r in rows}):
        idx = [i for i, r in enumerate(rows) if r['type'] == t]
        print(f'   {t:12s} CER = {cer_metric.compute(predictions=[preds[i] for i in idx], references=[refs[i] for i in idx]):.4f}  ({len(idx)})')
    for p, r in list(zip(preds, refs))[:10]:
        print('GT  :', r); print('PRED:', p); print()

evaluate_rows(val_rows, 'gold_val')
# evaluate_rows(read('test'), 'test')   # only once at the very end, never for tuning

gold_val: CER = 0.0905  (4333 crops)
   annotation   CER = 0.4015  (69)
   handwritten  CER = 0.0829  (4203)
   printed      CER = 0.4774  (61)
GT  : Магія голосу.
PRED: Магія голосу

GT  : Спочатку був "брехунець". З нього почи-
PRED: Спочатку був "брехунець". З нього почи-

GT  : нався день; клац - пронизливе рокотання
PRED: нався день: клац - і пронизливе рокотання

GT  : перших тактів "Реве та стогне", а за
PRED: перших тактів "Реве та стогне", а за

GT  : а за ним примусово-бадьоре "Говорить Київ!"
PRED: ним примусово - бадьоре "Говорить Коїв!

GT  : давали знак, що з-під теплої ковдри
PRED: давали знак, що з-під теплої ковдри

GT  : хоч-не-хоч пора вилізати
PRED: хоч-не-хоч пора вилізати.

GT  : Діти люблять чутися в гурті.
PRED: Діти люблять слутися в гурті.

GT  : Було приємно думати, що цієї миті по
PRED: Було приємно думати, що цієї миті по

GT  : країні мільйони дітей так само збирають-
PRED: граїні мільйони дітей так само збирають-



In [7]:
for t in ['annotation', 'printed']:
    rows = [r for r in val_rows if r['type'] == t]
    preds = recognize([Image.open(CROPS / r['path']).convert('RGB') for r in rows])
    scored = sorted(((cer_metric.compute(predictions=[p], references=[r['text']]), r['text'], p)
                     for r, p in zip(rows, preds)), reverse=True)
    print(f'\n=== {t}: 12 worst ===')
    for c, gt, p in scored[:12]:
        print(f'{c:.2f} | GT: {gt[:80]!r}\n     | PR: {p[:80]!r}')


=== annotation: 12 worst ===
4.50 | GT: '45'
     | PR: 'у в заціє'
1.00 | GT: 'і'
     | PR: ';'
1.00 | GT: '85/8'
     | PR: '05/ 19'
1.00 | GT: '3'
     | PR: '23'
1.00 | GT: '2'
     | PR: '1'
1.00 | GT: '12/6'
     | PR: 'Курь'
0.89 | GT: 'Справа 80 ар. в Любарі Швець'
     | PR: '3 5 же з'
0.80 | GT: '80 об'
     | PR: '300 "'
0.80 | GT: '10 зв'
     | PR: '10. світ'
0.74 | GT: 'назва хіміч речовин'
     | PR: 'хіхіч рагати 5б'
0.73 | GT: '± 10б. + 5б'
     | PR: '±5б'
0.72 | GT: 'хімічні реакції? типи реакцій'
     | PR: 'хімічні?'

=== printed: 12 worst ===
6.45 | GT: '2) показує переваги споживача'
     | PR: '2) (Illlllllllllllllllllllllllllllllllllllllllllllllllllllllllllllllllllllllllll'
3.48 | GT: 'ДТЕУ ФАКУЛЬТЕТ ФІНАНСІВ ТА ОБЛІКУ ДЕННО ФЕРМА НАВЧАННЯ'
     | PR: 'дена фiblblblblblblblblblblblblblblblblblblblblblblblblblblblblblblblblblblblblb'
1.16 | GT: '160° 140° 120° 100° 80° на захід від Гринвіча 0° на схід від Гринвіча 80° 100° 1'
     | PR: '180 зав" 120 100 во на